# Parameter efficient fine tuning on Deepseek model
## Parameter efficient fine tuning DeepSeek R1 on movie subtitles and summary pairs using LoRA adapter.
1. Data: trainingSubs.csv [movie_id, subtitles,summaries]
2. Model: Load base model in 4-bit quantization
3. LoRA adapters- 1% weights are trainable
4. Train- SFT
5. Saving the model
6. Reload merged model and generate summaries for the first few movies

# Install Dependencies 

In [3]:
import subprocess, sys

packages = [
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",
    "trl>=0.8.0",
    "accelerate>=0.27.0",
    "datasets>=2.18.0",
    "openai",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", pkg])

print("All packages installed.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.4 requires dill<0.4.2,>=0.3.0, which is not installed.
datasets 4.8.4 requires multiprocess<0.70.20, which is not installed.
datasets 4.8.4 requires pyarrow>=21.0.0, which is not installed.
datasets 4.8.4 requires xxhash, which is not installed.
peft 0.10.0 requires accelerate>=0.21.0, which is not installed.
peft 0.10.0 requires torch>=1.13.0, which is not installed.
sacrebleu 2.6.0 requires colorama, which is not installed.
trl 1.2.0 requires accelerate>=1.4.0, which is not installed.
ERROR: Operation cancelled by user


KeyboardInterrupt: 

# Configuration 

In [1]:
TRAIN_CSV_PATH = "./trainingSubs.csv"
TEST_CSV_PATH  = "./testSubs.csv"
LLM_api_key="sk-g6Y1MHJaD_y1bohW7m_ItA"
LLM_base_url="https://ol.sci.pitt.edu"
OUTPUT_DIR = "./finetuned_deepseekr1_model"
MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B" 
LORA_RANK    = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

NUM_EPOCHS     = 3
BATCH_SIZE     = 2
GRAD_ACCUM     = 8
LEARNING_RATE  = 5e-5
MAX_SEQ_LENGTH = 1024


INSTRUCTION = "Given these movie subtitles, write a faithful plot summary in one paragraph. Use only information supported by the subtitles."


In [1]:
from openai import OpenAI
from transformers import AutoTokenizer
import pandas as pd
from datasets import Dataset


KeyboardInterrupt



KeyboardInterrupt: 

# Load and Format Training Data

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer ready")
print("Tokenizer model_max_length:", getattr(tokenizer, "model_max_length", "unknown"))

Tokenizer ready
Tokenizer model_max_length: 16384


In [5]:
import pandas as pd
from datasets import Dataset

train_df = pd.read_csv(TRAIN_CSV_PATH).dropna(subset=["subtitles", "summaries"]).copy()
train_df["subtitles"] = train_df["subtitles"].astype(str)
train_df["summaries"] = train_df["summaries"].astype(str).str.strip()
train_df = train_df[train_df["summaries"].str.len() > 0].reset_index(drop=True)
print(f"Train: {len(train_df)} rows")

def get_subtitle_excerpt(subtitle, max_chars=2000):
    subtitle = str(subtitle)
    if len(subtitle) <= max_chars:
        return subtitle
    half = max_chars // 2
    return subtitle[:half] + "\n...\n" + subtitle[-half:]

def format_training_example(subtitle, summary):
    return (
        f"### Instruction:\n{INSTRUCTION}\n\n"
        f"### Subtitles:\n{get_subtitle_excerpt(subtitle)}\n\n"
        f"### Summary:\n{str(summary).strip()}"
    )

train_texts = [format_training_example(row["subtitles"], row["summaries"]) for _, row in train_df.iterrows()]
train_dataset = Dataset.from_dict({"text": train_texts})
print(f"Training examples: {len(train_dataset)}")

Train: 832 rows
Training examples: 832


# Load base model and tokenizer

In [6]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

compute_dtype = torch.bfloat16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)

model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

print(f"Model loaded. Parameters: {model.num_parameters():,}")
print("Compute dtype:", compute_dtype)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/ihome/infsci2440-2026s/rab527/.local/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model loaded. Parameters: 8,030,261,248
Compute dtype: torch.bfloat16


# LoRA Adapaters

In [13]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}  ({100 * trainable / total:.2f}%)")

Trainable: 41,943,040  (0.92%)


# SFT training and prompt masking

In [14]:
from datasets import Dataset

summary_marker = "### Summary:\n"
summary_ids = tokenizer.encode(summary_marker, add_special_tokens=False)

def find_subsequence(sequence, subsequence):
    if not subsequence:
        return -1
    last_start = len(sequence) - len(subsequence) + 1
    for i in range(max(0, last_start)):
        if sequence[i:i + len(subsequence)] == subsequence:
            return i
    return -1

# masking cell - NO padding here, let the collator handle it
def mask_prompt(example):
    input_ids = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
    )["input_ids"]
    labels = input_ids.copy()

    mask_end = 0
    for i in range(len(input_ids) - len(summary_ids)):
        if input_ids[i : i + len(summary_ids)] == summary_ids:
            mask_end = i + len(summary_ids)
            break

    labels[:mask_end] = [-100] * mask_end
    return {"input_ids": input_ids, "labels": labels, "attention_mask": [1] * len(input_ids)}

train_dataset_masked = train_dataset.map(mask_prompt, remove_columns=["text"])
print(f"Masked dataset ready: {len(train_dataset_masked)} examples")

Map:   0%|          | 0/832 [00:00<?, ? examples/s]

Masked dataset ready: 832 examples


# Train

In [15]:
# training cell - back to SFTTrainer, no DataCollator
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./model_deepseek_checkpoint",
    overwrite_output_dir=True,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    max_grad_norm=0.3,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_masked,
    processing_class=tokenizer,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss
10,3.030500
20,2.807900
30,2.668300
40,2.628900
50,2.674400
60,2.567900
70,2.559400
80,2.493200
90,2.519700
100,2.542300


TrainOutput(global_step=156, training_loss=2.5853129839285827, metrics={'train_runtime': 4995.7744, 'train_samples_per_second': 0.5, 'train_steps_per_second': 0.031, 'total_flos': 1.1049447504563405e+17, 'train_loss': 2.5853129839285827})

# Save model

In [23]:
import os, json

merged_model = model.merge_and_unload()
os.makedirs(OUTPUT_DIR, exist_ok=True)
merged_model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

json.dump(
    {
        "base_model": MODEL_NAME,
        "fine_tuning": "LoRA (merged)",
        "lora_rank": LORA_RANK,
        "epochs": NUM_EPOCHS,
        "train_examples": len(train_dataset),
        "think_suppression": "empty <think></think> block in training format"
    },
    open(os.path.join(OUTPUT_DIR, "model_info.json"), "w"),
    indent=2
)

print(f"Merged model saved to: {OUTPUT_DIR}")

Merged model saved to: ./finetuned_deepseekr1_model
